# <center>**Travaux exploratoires : résumé formaté d'un texte</center>**

# <center>**VI. Fonction d'extraction structurée : version complète</center>**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 06/08/2025*

**Contexte :**

  - dans le notebook *04_modele_llama3-70b-8192.ipynb* on a codé une fonction *extraction_structuree(nom_fichier)* qui en entrée prend le nom d'un fichier texte, et en sortie crée et enregistre un fichier .csv et un fichier .json qui stockent des extractions structurées des événements présents dans le fichier, sous la forme

    - résumé (5-6 mots)

    - lieu

    - moment

    - individus

    Cette fonction est en fait une surcouche d'une fonction extraire_faits(), présente dans le même notebook, qui à partir d'un texte relatant un événement rend en sortie l'extraction structurée décrite ci-dessus.

  - dans le notebook *05_decoupage_texte.ipynb* prend en entrée le nom d'un fichier texte et le nom d'un répertoire, et rend en sortie des blocs de ce texte au format .json, dans le répertoire donné en entrée.


**Méthodologie :** elle consiste à implémenter le **pipeline** suivant :

- lecture d'un fichier texte
- découpage du fichier texte en blocs
- regroupement des blocs en batchs
- appel au modèle pour extraire les faits
- sauvegarde des résultats au fur et à mesure (checkpoint)
- production d' un export final CSV et JSON

**Focus sur le découpage du fichier en blocs + le regroupement des blocs en batchs :**

Fichier texte complet

       │
       ▼

decouper_texte()

       │
       ▼

Liste de blocs

[ bloc0, bloc1, bloc2, bloc3, bloc4, ... ]

       │
       ▼

extraire_faits_depuis_texte_long_batch()

       │
       ├── Batch 1 : bloc0 à bloc7
       ├── Batch 2 : bloc8 à bloc15
       └── Batch 3 : bloc16 à bloc...


**Améliorations possibles :**

- améliorer la robustesse de decouper_texte()

- llama-3.3-70b-versatile est très puissant mais aussi lent pour de gros batchs → penser à tester llama-3.1-8b-instant pour un traitement rapide (moins cher et QPS plus élevé).

- augmenter qps pour que ça aille plus vite, mais surveiller les limites Groq.



**Résultats :**


**Conclusion :**

# **I. Pipeline complet de l'extraction structurée d'événements**

On utilise le client OpenAI pointé vers Groq (base_url="https://api.groq.com/openai/v1"), avec la clé dans GROQ_API_KEY.

**Objectif :** réduire le nombre d’appels réseau en envoyant plusieurs blocs de texte en une seule requête (batch).

**Pipeline complète :**

<center>découpe du fichier → batching → appel modèle → checkpoint → export CSV/JSON</center>

**Import des librairies**

In [2]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 4.0 MB/s eta 0:00:00


In [3]:
import pandas as pd
from openai import OpenAI
import time
import json
import os
import re
import math
from typing import Dict, List, Any
from groq import Groq

In [4]:
!python -m spacy download fr_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 77.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('fr_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


**Initialisation du client**

On importe le fichier .txt contenant la clé *groq_key*. Ici on suppose que cette clé est stockée dans un fichier *groq_key.txt*

In [5]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier groq_key.txt

Saving groq_key.txt to groq_key.txt


**Configuration du client OpenAI pour l'API Groq**

In [6]:
# 🔑 clé Groq
#  Bonne pratique : lire la clé depuis un fichier texte
with open("groq_key.txt", "r", encoding="utf-8") as f:
    GROQ_API_KEY = f.read().strip()

# 📍 Configuration du client OpenAI pour l'API Groq
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
    default_headers={"User-Agent": "pipeline-faits/1.0"})

In [7]:
def decouper_texte(nom_fichier, target_chars=2000, overlap=200, min_chars=400):
    with open(nom_fichier, "r", encoding="utf-8") as f:
        texte = f.read()
    morceaux, i, n = [], 0, len(texte)
    while i < n:
        j = min(i + target_chars, n)
        chunk = texte[i:j]
        if j < n:
            fin = max(chunk.rfind(". "), chunk.rfind("? "), chunk.rfind("! "))
            if fin > min_chars:
                chunk = chunk[:fin+1]
                j = i + len(chunk)
        if len(chunk) >= min_chars:
            morceaux.append(chunk)
        i = max(j - overlap, j)
    return morceaux

**Fonctions utiles**

Parse TSV + marqueurs

In [59]:
import re

def _extract_between_markers(text: str, start="BEGIN_TSV", end="END_TSV") -> str:
    m = re.search(rf"{re.escape(start)}\s*(.*?)\s*{re.escape(end)}", text, flags=re.S)
    return m.group(1).strip() if m else ""

def _parse_tsv_lines_with_markers(blob: str):
    """
    Récupère le bloc entre BEGIN_TSV / END_TSV,
    puis parse des lignes TSV → 4 colonnes: resume, lieu, moment, individus.
    Ignore les lignes qui n'ont pas 4 colonnes.
    """
    inner = _extract_between_markers(blob, "BEGIN_TSV", "END_TSV")
    if not inner:
        return []

    rows = []
    for raw in inner.splitlines():
        line = raw.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) < 4:
            continue
        parts = parts[:4]
        rows.append({
            "resume":    re.sub(r"\s+", " ", parts[0].strip()),
            "lieu":      re.sub(r"\s+", " ", parts[1].strip()),
            "moment":    re.sub(r"\s+", " ", parts[2].strip()),
            "individus": re.sub(r"\s+", " ", parts[3].strip()),
        })
    return rows

In [8]:
import re, json, time
import pandas as pd

def _strip_code_fences(txt: str) -> str:
    return re.sub(r"^```[^\n]*\n|\n```$", "", txt.strip())

def _parse_csv_lines(blob: str):
    """
    Attend uniquement des lignes CSV sans guillemets ni entête, 4 colonnes :
    resume,lieu,moment,individus
    - ignore toute ligne qui n'a pas exactement 3 virgules
    - trim + normalisation espaces
    """
    rows = []
    for raw in _strip_code_fences(blob).splitlines():
        line = raw.strip()
        if not line or line.lower().replace(" ", "") == "resume,lieu,moment,individus":
            continue
        # EXACTEMENT 3 virgules → 4 champs
        if line.count(",") != 3:
            continue
        parts = [p.strip() for p in line.split(",", 3)]
        # nettoyage doux (pas de tab, espaces multipls)
        parts = [re.sub(r"\s+", " ", p.replace("\t", " ").strip()) for p in parts]
        rows.append({
            "resume": parts[0],
            "lieu": parts[1],
            "moment": parts[2],
            "individus": parts[3],
        })
    return rows

**Appel au LLM**

In [60]:
rules_param = (
    "RENVOIE UNIQUEMENT un bloc entre ces marqueurs, sans rien d'autre :\n"
    "BEGIN_TSV\n"
    "(lignes TSV)\n"
    "END_TSV\n"
    "\n"
    "Format : lignes TSV, une ligne par fait, EXACTEMENT 4 colonnes dans cet ordre :\n"
    "resume\tlieu\tmoment\tindividus\n"
    "\n"
    "DÉFINITIONS :\n"
    "- resume : résumé très bref (5 à 12 mots), décrivant l’action principale. "
    "  La colonne resume ne mentionne aucun lieu, aucun moment, aucun individu.\n"
    "- lieu : lieu où l'action s'est déroulée (adresse, nom de lieu, ville) ; sinon vide.\n"
    "- moment : utiliser le schéma 'DATE - HEURE' si dispo. "
    "  • Si seule la date est connue : 'DATE - '\n"
    "  • Si seule l'heure est connue : ' - HEURE'\n"
    "  • Si un moment textuel existe ('le matin', 'au crépuscule', 'hier soir', 'pendant la cérémonie'), "
    "    l’insérer dans la partie manquante (ex. ' - le matin', '05 mai 2025 - le soir'). "
    "  • Si aucun repère temporel : moment vide.\n"
    "- individus : personnes impliquées, séparées par '; ' (noms complets si connus ; sinon 'homme inconnu', etc.) ; peut être vide.\n"
    "\n"
    "CONTRAINTES :\n"
    "- Extraire TOUS les faits distincts, même mineurs ; une ligne par fait ; ne pas fusionner.\n"
    "- Ne pas mélanger les informations entre colonnes.\n"
    "- Toujours produire 4 colonnes séparées par TAB ; colonnes vides autorisées.\n"
    "- INTERDIT : virgules pour séparer les colonnes, guillemets, Markdown, texte hors des marqueurs.\n"
    "- Remplacer toute tabulation/retour de ligne interne à une cellule par un espace.\n"
    "\n"
    "EXEMPLES :\n"
    "BEGIN_TSV\n"
    "Échange devant café Le Nautilus\t25 rue de Crimée 75019 Paris\t02 mai 2025 - 08h47\tClaire Dubois; Julien Morel\n"
    "Visite chez Claire\t18 rue de Charonne 75011 Paris\t04 mai 2025 - 15h15\tClaire Dubois; Fatima El-Haddad\n"
    "Discussion près de la gare\tGare de Lyon 75012 Paris\t - en fin d’après-midi\tThomas Rivière\n"
    "Départ précipité\t\t10 mai 2025 - 23h48\tFatima El-Haddad\n"
    "Observation discrète\t\t - \t\n"
    "END_TSV\n"
)

In [61]:
# Modèles à essayer (en priorité)
MODEL_CANDIDATES = [
    "llama-3.1-8b-instant",     # 500K TPD
    "llama-3.3-70b-versatile",  # qualité
]

def _llm_extract_tsv(items, model, max_tokens=1200, temperature=0.1):
    messages = [
        {"role": "system", "content": "Tu es un extracteur d'événements, exhaustif et strict sur le format."},
        {"role": "user", "content": rules_param + "\n\nTEXTE À ANALYSER (génère une ou plusieurs lignes par fait détecté) :\n"
                          + json.dumps({"items":[{"id":it["id"],"texte":it["texte"]} for it in items]}, ensure_ascii=False)}
    ]
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content if resp.choices else ""

**Fonction extraction_structuree()**

Il s'agit du chef d'orchestre de ce notebook.

In [62]:
def extraction_structuree(
    nom_fichier: str,
    batch_size: int = 2,
    model: str = "llama-3.1-8b-instant",
    target_chars: int = 2000,
    overlap: int = 200,
    min_chars: int = 400,
    out_csv: str = "faits_extraits.csv",
    checkpoint_csv: str = "checkpoint_faits.csv",
    max_completion_tokens: int = 1200,
    temperature: float = 0.1,
    qps: float = 0.4,           # ≈ 24 RPM
    keep_checkpoint: bool = False,
):
    """
    Lit un .txt, le découpe en blocs, fait des extractions par batchs,
    et écrit un CSV final avec 4 colonnes: resume, lieu, moment, individus.
    'moment' peut être 'DATE - HEURE', 'DATE - texte', ' - texte', ou vide.
    """
    blocs = decouper_texte(nom_fichier, target_chars, overlap, min_chars)
    print(f"✅ {len(blocs)} blocs détectés.")
    if not blocs:
        df0 = pd.DataFrame(columns=["resume","lieu","moment","individus"])
        df0.to_csv(out_csv, index=False, encoding="utf-8")
        print(f"💾 {out_csv} écrit (0 lignes).")
        return df0

    # checkpoint
    if not keep_checkpoint and os.path.exists(checkpoint_csv):
        try: os.remove(checkpoint_csv)
        except Exception: pass
    header_written = False
    def write_checkpoint(rows):
        nonlocal header_written
        if not rows: return
        df_ck = pd.DataFrame(rows, columns=["resume","lieu","moment","individus"])
        df_ck.to_csv(
            checkpoint_csv,
            mode="a",
            index=False,
            header=(not header_written and (not os.path.exists(checkpoint_csv))),
            encoding="utf-8"
        )
        header_written = True

    lignes = []
    last_call = 0.0

    # boucle batchs
    for start in range(0, len(blocs), batch_size):
        end = min(start + batch_size, len(blocs))
        batch_items = [{"id": i, "texte": blocs[i].strip()} for i in range(start, end)]

        # QPS simple
        now = time.time()
        delta = 1.0 / qps - (now - last_call)
        if delta > 0: time.sleep(delta)
        last_call = time.time()

        # Appel principal (on essaie d'abord 'model', sinon autres candidats)
        text_out = ""
        last_err = None
        for m in [model] + [mm for mm in MODEL_CANDIDATES if mm != model]:
            try:
                text_out = _llm_extract_tsv(batch_items, m, max_tokens=max_completion_tokens, temperature=temperature)
                _save_debug(text_out, f"batch_{start+1}_{end}")
                if text_out:
                    break
            except Exception as e:
                last_err = e
                continue

        if not text_out and last_err:
            print(f"⚠️ Erreur batch {start+1}-{end}: {last_err}")

        # Parse TSV
        new_rows = _parse_tsv_lines_with_markers(text_out)

        # Plan B : si le batch renvoie peu/néant, on traite item par item
        if (not new_rows) and len(batch_items) > 1:
            print("🧩 Réponse vide/insuffisante sur le batch ⇒ bascule en singletons.")
            for it in batch_items:
                try:
                    one = _llm_extract_tsv([it], model, max_tokens=min(1400, max_completion_tokens+200), temperature=temperature)
                    _save_debug(one, f"item_{it['id']}")
                except Exception as e:
                    print(f"❌ Item {it['id']} en échec: {str(e)[:120]}")
                    one = ""
                new_rows.extend(_parse_tsv_lines_with_markers(one))

        # Accumule + checkpoint
        before = len(lignes)
        lignes.extend(new_rows)
        write_checkpoint(lignes[before:])

        # Progress
        pct = 100.0 * end / len(blocs)
        print(f"Batch {start+1}-{end} traité | {end}/{len(blocs)} blocs ({pct:.1f}%) – {len(new_rows)} faits")

    # Export final
    df = pd.DataFrame(lignes, columns=["resume","lieu","moment","individus"])
    df.to_csv(out_csv, index=False, encoding="utf-8")
    print(f"💾 {out_csv} écrit ({len(df)} lignes).")
    return df

**Application à un compte-rendu d'enquête simulé**

In [11]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier CRE.txt

Saving CRE.txt to CRE.txt


In [63]:
import time

start_time = time.time()

df = extraction_structuree(
    nom_fichier="CRE.txt",   # ton fichier à analyser
    batch_size=2,                    # petit batch = moins de coupures
    model="llama-3.1-8b-instant",    # modèle TPD élevé
    target_chars=2000,
    overlap=200,
    min_chars=400,
    out_csv="faits_extraits.csv",    # CSV final
    checkpoint_csv="checkpoint_faits.csv",
    qps=0.4,
    max_completion_tokens=1000,
    temperature=0.1
)

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")
print(df.head())  # aperçu

✅ 4 blocs détectés.
Batch 1-2 traité | 2/4 blocs (50.0%) – 8 faits
Batch 3-4 traité | 4/4 blocs (100.0%) – 12 faits
💾 faits_extraits.csv écrit (20 lignes).
⏱️ Temps d'exécution : 5.10 secondes
                       resume                                        lieu  \
0         Échange devant café                25 rue de Crimée 75019 Paris   
1          Visite chez Claire              18 rue de Charonne 75011 Paris   
2  Conversation au restaurant  42 rue du Faubourg Saint-Denis 75010 Paris   
3            Départ précipité      5 rue de la République 93100 Montreuil   
4           Montée à bord TGV                    Gare de Lyon 75012 Paris   

                moment                         individus  
0  02 mai 2025 - 08h47       Claire Dubois; Julien Morel  
1  04 mai 2025 - 15h15   Claire Dubois; Fatima El-Haddad  
2  07 mai 2025 - 19h02      Julien Morel; Thomas Rivière  
3  10 mai 2025 - 23h48                  Fatima El-Haddad  
4  11 mai 2025 - 10h11  Thomas Rivière; Fatima El

In [64]:
df

,resume,lieu,moment,individus
0,Échange devant café,25 rue de Crimée 75019 Paris,02 mai 2025 - 08h47,Claire Dubois; Julien Morel
1,Visite chez Claire,18 rue de Charonne 75011 Paris,04 mai 2025 - 15h15,Claire Dubois; Fatima El-Haddad
2,Conversation au restaurant,42 rue du Faubourg Saint-Denis 75010 Paris,07 mai 2025 - 19h02,Julien Morel; Thomas Rivière
3,Départ précipité,5 rue de la République 93100 Montreuil,10 mai 2025 - 23h48,Fatima El-Haddad
4,Montée à bord TGV,Gare de Lyon 75012 Paris,11 mai 2025 - 10h11,Thomas Rivière; Fatima El-Haddad
5,Discussion tendue,7 rue du Plat 69002 Lyon,11 mai 2025 - 14h42,Thomas Rivière; Fatima El-Haddad
6,Appel téléphonique,18 rue de Charonne 75011 Paris,14 mai 2025 - 18h20,Claire Dubois
7,Interpellation pour audition,42 rue du Faubourg Saint-Denis 75010 Paris,16 mai 2025 - 09h05,Julien Morel
8,Présence de femmes filmée,"25 rue de Crimée, 75019 Paris",18 mai 2025 - 20h33,Claire Dubois; Fatima El-Haddad
9,Sortie de cabine téléphonique,"102 avenue de la République, 75011 Paris",21 mai 2025 - 12h14,Julien Morel


**Avec un autre compte-rendu d'enquête simulé, mais au style un peu plus libre**

In [66]:
from google.colab import files

uploaded = files.upload() # on upolad le fichier CRE02.txt

Saving CRE02.txt to CRE02.txt


In [67]:
import time

start_time = time.time()

df = extraction_structuree(
    nom_fichier="CRE02.txt",   # ton fichier à analyser
    batch_size=2,                    # petit batch = moins de coupures
    model="llama-3.1-8b-instant",    # modèle TPD élevé
    target_chars=2000,
    overlap=200,
    min_chars=400,
    out_csv="faits_extraits.csv",    # CSV final
    checkpoint_csv="checkpoint_faits.csv",
    qps=0.4,
    max_completion_tokens=1000,
    temperature=0.1
)

end_time = time.time()
print(f"⏱️ Temps d'exécution : {end_time - start_time:.2f} secondes")
print(df.head())  # aperçu

✅ 5 blocs détectés.
Batch 1-2 traité | 2/5 blocs (40.0%) – 22 faits
Batch 3-4 traité | 4/5 blocs (80.0%) – 28 faits
Batch 5-5 traité | 5/5 blocs (100.0%) – 4 faits
💾 faits_extraits.csv écrit (54 lignes).
⏱️ Temps d'exécution : 9.32 secondes
                                resume  \
0                  Découverte du corps   
1         Identification de la victime   
2       Déclaration de Sophie Lemaitre   
3  Discussion entre Lucien et un homme   
4  Présence de Lucien et d’un individu   

                                           lieu               moment  \
0              38 quai de la Loire, 75019 Paris  03 mai 2025 - 07h45   
1              38 quai de la Loire, 75019 Paris  03 mai 2025 - 07h45   
2              38 quai de la Loire, 75019 Paris  03 mai 2025 - 07h45   
3  Près de l’entrée du parc des Buttes-Chaumont  02 mai 2025 - 23h15   
4                          avenue Simon Bolivar  02 mai 2025 - 23h12   

                                           individus  
0  Capitaine Marc 

In [68]:
df

,resume,lieu,moment,individus
0,Découverte du corps,"38 quai de la Loire, 75019 Paris",03 mai 2025 - 07h45,Capitaine Marc Delorme; Claire Morin; Lucien B...
1,Identification de la victime,"38 quai de la Loire, 75019 Paris",03 mai 2025 - 07h45,Capitaine Marc Delorme; Claire Morin; Lucien B...
2,Déclaration de Sophie Lemaitre,"38 quai de la Loire, 75019 Paris",03 mai 2025 - 07h45,Sophie Lemaitre; Capitaine Marc Delorme; Clair...
3,Discussion entre Lucien et un homme,Près de l’entrée du parc des Buttes-Chaumont,02 mai 2025 - 23h15,Lucien Borel; homme inconnu
4,Présence de Lucien et d’un individu,avenue Simon Bolivar,02 mai 2025 - 23h12,Lucien Borel; homme inconnu
5,Sortie du Bistrot du Canal,12 rue de Meaux,02 mai 2025 - 22h30,Lucien Borel; Karim Haddad
6,Passage à la station Jaurès,Station Jaurès,02 mai 2025 - 22h41,Lucien Borel
7,Interrogatoire de Karim,,-,Karim Haddad; Capitaine Marc Delorme
8,Déclaration de Paul Dervaux,"5 rue des Maraîchers, Montreuil",-,Paul Dervaux; Capitaine Marc Delorme
9,Analyse de l’autopsie,,-,Lucien Borel
